# Pull all ChEMBL J05 antivirals and prepare pipeline datasets
This notebook pulls all molecules in the J05 antiviral class from ChEMBL, standardizes the core fields, and saves datasets for the rest of the pipeline.

In [1]:
# Install if needed
# !pip install chembl_webresource_client pandas rdkit-pypi requests
# !pip install rdkit-pypi

In [2]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

try:
    from rdkit import Chem
    from rdkit.Chem import Descriptors, Crippen, Lipinski
    RDKIT_AVAILABLE = True
except ImportError:
    RDKIT_AVAILABLE = False
    print("RDKit is not installed. Run: !pip install rdkit-pypi")

In [3]:
# Create API
molecule = new_client.molecule
mechanism = new_client.mechanism
drug_indication = new_client.drug_indication

### 1) Pull all J05 antiviral molecules

This collects the full J05 antiviral set. If you want to pull only approved drugs, add the fileter `max_phase=4`, right now it pulls all antivirals for the training set.

In [6]:
# Pull all molecules that have an ATC classification beginning with J05 (antivirals)

antivirals = list(
    molecule.filter(atc_classifications__level5__startswith="J05")
    # max_phase=4
)

print(f"Retrieved {len(antivirals)} ChEMBL records from J05.")
if antivirals:
    print("Example keys:", list(antivirals[0].keys())[:15])

Retrieved 89 ChEMBL records from J05.
Example keys: ['atc_classifications', 'availability_type', 'biotherapeutic', 'black_box_warning', 'chemical_probe', 'chirality', 'cross_references', 'dosed_ingredient', 'first_approval', 'first_in_class', 'helm_notation', 'inorganic_flag', 'max_phase', 'molecule_chembl_id', 'molecule_hierarchy']


In [7]:
# Flatten J05 antiviral records from ChEMBL into a dataframe

records = []

for mol in antivirals:
    structures = mol.get("molecule_structures") or {}
    props = mol.get("molecule_properties") or {}
    synonyms = mol.get("molecule_synonyms") or []
    atc = mol.get("atc_classifications") or []

    # Safely collect ATC values whether returned as dicts or strings
    atc_values = []
    for item in atc:
        if isinstance(item, dict):
            atc_values.extend([v for v in item.values() if isinstance(v, str)])
        elif isinstance(item, str):
            atc_values.append(item)

    # Safely collect synonym strings
    synonym_values = []
    for s in synonyms:
        if isinstance(s, dict):
            val = s.get("molecule_synonym")
            if val:
                synonym_values.append(val)
        elif isinstance(s, str):
            synonym_values.append(s)

    records.append({
        "chembl_id": mol.get("molecule_chembl_id"),
        "pref_name": mol.get("pref_name"),
        "molecule_type": mol.get("molecule_type"),
        "max_phase": mol.get("max_phase"),
        "therapeutic_flag": mol.get("therapeutic_flag"),
        "canonical_smiles": structures.get("canonical_smiles"),
        "standard_inchi_key": structures.get("standard_inchi_key"),
        "full_mwt": props.get("full_mwt"),
        "alogp": props.get("alogp"),
        "hba": props.get("hba"),
        "hbd": props.get("hbd"),
        "num_ro5_violations": props.get("num_ro5_violations"),
        "synonyms": "; ".join(sorted(set(synonym_values))),
        "atc_codes": "; ".join(sorted(set(atc_values)))
    })

raw_df = pd.DataFrame(records)

print("Raw dataframe shape:", raw_df.shape)
display(raw_df.head())

Raw dataframe shape: (89, 14)


,chembl_id,pref_name,molecule_type,max_phase,therapeutic_flag,canonical_smiles,standard_inchi_key,full_mwt,alogp,hba,hbd,num_ro5_violations,synonyms,atc_codes
0,CHEMBL57,NEVIRAPINE,Small molecule,4.0,True,Cc1ccnc2c1NC(=O)c1cccnc1N2C1CC1,NQDJXKOVJZTUJA-UHFFFAOYSA-N,266.30,2.65,4.0,1.0,0.0,BIRG 0587; BIRG-0587; BIRG0587; NSC-641530; Ne...,J05AG01
1,CHEMBL114,SAQUINAVIR,Small molecule,4.0,True,CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C...,QWAXKHKRTORLEM-UGJKXSETSA-N,670.86,3.09,7.0,5.0,1.0,Fortovase; RO 31-8959/000; RO-31-8959/000; SCH...,J05AE01
2,CHEMBL584,NELFINAVIR,Small molecule,4.0,True,Cc1c(O)cccc1C(=O)N[C@@H](CSc1ccccc1)[C@H](O)CN...,QAGYKUNXZHXKMR-HKWSIXNMSA-N,567.80,4.75,6.0,4.0,1.0,NSC-747167; Nelfinavir,J05AE04
3,CHEMBL116,AMPRENAVIR,Small molecule,4.0,True,CC(C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O[C@H]...,YMARZQAQMVYCKC-OEMFJLHTSA-N,505.64,2.40,7.0,3.0,1.0,141 W94; 141W94; Agenerase; Amprenavir; J05AE0...,J05AE05
4,CHEMBL593,DELAVIRDINE,Small molecule,4.0,True,CC(C)Nc1cccnc1N1CCN(C(=O)c2cc3cc(NS(C)(=O)=O)c...,WHBIGIKBNXZKFE-UHFFFAOYSA-N,456.57,2.72,6.0,3.0,0.0,Delavirdina; Delavirdine; U-90152,J05AG02


### 2) Basic cleaning for the raw molecular table

In [8]:
# Keep rows with both ChEMBL ID and SMILES
df = raw_df.copy()
df = df.dropna(subset=["chembl_id", "canonical_smiles"]).drop_duplicates(subset=["chembl_id"]).reset_index(drop=True)

print(f"Rows after removing missing IDs/SMILES and duplicate ChEMBL IDs: {len(df)}")
df.head()

Rows after removing missing IDs/SMILES and duplicate ChEMBL IDs: 87


,chembl_id,pref_name,molecule_type,max_phase,therapeutic_flag,canonical_smiles,standard_inchi_key,full_mwt,alogp,hba,hbd,num_ro5_violations,synonyms,atc_codes
0,CHEMBL57,NEVIRAPINE,Small molecule,4.0,True,Cc1ccnc2c1NC(=O)c1cccnc1N2C1CC1,NQDJXKOVJZTUJA-UHFFFAOYSA-N,266.30,2.65,4.0,1.0,0.0,BIRG 0587; BIRG-0587; BIRG0587; NSC-641530; Ne...,J05AG01
1,CHEMBL114,SAQUINAVIR,Small molecule,4.0,True,CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C...,QWAXKHKRTORLEM-UGJKXSETSA-N,670.86,3.09,7.0,5.0,1.0,Fortovase; RO 31-8959/000; RO-31-8959/000; SCH...,J05AE01
2,CHEMBL584,NELFINAVIR,Small molecule,4.0,True,Cc1c(O)cccc1C(=O)N[C@@H](CSc1ccccc1)[C@H](O)CN...,QAGYKUNXZHXKMR-HKWSIXNMSA-N,567.80,4.75,6.0,4.0,1.0,NSC-747167; Nelfinavir,J05AE04
3,CHEMBL116,AMPRENAVIR,Small molecule,4.0,True,CC(C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O[C@H]...,YMARZQAQMVYCKC-OEMFJLHTSA-N,505.64,2.40,7.0,3.0,1.0,141 W94; 141W94; Agenerase; Amprenavir; J05AE0...,J05AE05
4,CHEMBL593,DELAVIRDINE,Small molecule,4.0,True,CC(C)Nc1cccnc1N1CCN(C(=O)c2cc3cc(NS(C)(=O)=O)c...,WHBIGIKBNXZKFE-UHFFFAOYSA-N,456.57,2.72,6.0,3.0,0.0,Delavirdina; Delavirdine; U-90152,J05AG02


### 3) RDKit standardization and descriptor calculation

This creates the first cleaned dataset for the pipeline with valid, SMILES plus a few descriptors and rule-based filters.

In [9]:
def standardize_smiles(smiles: str):
    if not RDKIT_AVAILABLE or pd.isna(smiles):
        return None, False, None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None, False, None
        canonical = Chem.MolToSmiles(mol, canonical=True)
        return canonical, True, mol
    except Exception:
        return None, False, None

cleaned = []
for _, row in df.iterrows():
    canon, valid, mol = standardize_smiles(row["canonical_smiles"])
    rec = row.to_dict()
    rec["rdkit_valid"] = valid
    rec["canonical_smiles_rdkit"] = canon

    if valid and mol is not None:
        rec["mw_rdkit"] = Descriptors.MolWt(mol)
        rec["logp_rdkit"] = Crippen.MolLogP(mol)
        rec["hba_rdkit"] = Lipinski.NumHAcceptors(mol)
        rec["hbd_rdkit"] = Lipinski.NumHDonors(mol)
        rec["rotatable_bonds"] = Lipinski.NumRotatableBonds(mol)
        rec["tpsa"] = Descriptors.TPSA(mol)
    else:
        rec["mw_rdkit"] = None
        rec["logp_rdkit"] = None
        rec["hba_rdkit"] = None
        rec["hbd_rdkit"] = None
        rec["rotatable_bonds"] = None
        rec["tpsa"] = None

    cleaned.append(rec)

clean_df = pd.DataFrame(cleaned)
clean_df = clean_df[clean_df["rdkit_valid"]].copy()
clean_df = clean_df.drop_duplicates(subset=["canonical_smiles_rdkit"]).reset_index(drop=True)

print(f"Rows after RDKit validation and duplicate canonical SMILES removal: {len(clean_df)}")
clean_df.head()

Rows after RDKit validation and duplicate canonical SMILES removal: 87


,chembl_id,pref_name,molecule_type,max_phase,therapeutic_flag,canonical_smiles,standard_inchi_key,full_mwt,alogp,hba,...,synonyms,atc_codes,rdkit_valid,canonical_smiles_rdkit,mw_rdkit,logp_rdkit,hba_rdkit,hbd_rdkit,rotatable_bonds,tpsa
0,CHEMBL57,NEVIRAPINE,Small molecule,4.0,True,Cc1ccnc2c1NC(=O)c1cccnc1N2C1CC1,NQDJXKOVJZTUJA-UHFFFAOYSA-N,266.30,2.65,4.0,...,BIRG 0587; BIRG-0587; BIRG0587; NSC-641530; Ne...,J05AG01,True,Cc1ccnc2c1NC(=O)c1cccnc1N2C1CC1,266.304,2.65122,4,1,1,58.12
1,CHEMBL114,SAQUINAVIR,Small molecule,4.0,True,CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C...,QWAXKHKRTORLEM-UGJKXSETSA-N,670.86,3.09,7.0,...,Fortovase; RO 31-8959/000; RO-31-8959/000; SCH...,J05AE01,True,CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C...,670.855,3.09240,7,5,12,166.75
2,CHEMBL584,NELFINAVIR,Small molecule,4.0,True,Cc1c(O)cccc1C(=O)N[C@@H](CSc1ccccc1)[C@H](O)CN...,QAGYKUNXZHXKMR-HKWSIXNMSA-N,567.80,4.75,6.0,...,NSC-747167; Nelfinavir,J05AE04,True,Cc1c(O)cccc1C(=O)N[C@@H](CSc1ccccc1)[C@H](O)CN...,567.796,4.74762,6,4,9,101.90
3,CHEMBL116,AMPRENAVIR,Small molecule,4.0,True,CC(C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O[C@H]...,YMARZQAQMVYCKC-OEMFJLHTSA-N,505.64,2.40,7.0,...,141 W94; 141W94; Agenerase; Amprenavir; J05AE0...,J05AE05,True,CC(C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O[C@H]...,505.637,2.40280,7,3,11,131.19
4,CHEMBL593,DELAVIRDINE,Small molecule,4.0,True,CC(C)Nc1cccnc1N1CCN(C(=O)c2cc3cc(NS(C)(=O)=O)c...,WHBIGIKBNXZKFE-UHFFFAOYSA-N,456.57,2.72,6.0,...,Delavirdina; Delavirdine; U-90152,J05AG02,True,CC(C)Nc1cccnc1N1CCN(C(=O)c2cc3cc(NS(C)(=O)=O)c...,456.572,2.71710,6,3,6,110.43


In [10]:
# Create a copy of the cleaned dataframe so we don't modify the original
filtered_df = clean_df.copy()

# Create a new column that checks if molecular weight is within a reasonable range (100–900)
# .between() returns True if the value is within the range
filtered_df["passes_mw_filter"] = filtered_df["mw_rdkit"].between(100, 900, inclusive="both")

# Create a new column that checks Lipinski's Rule of Five conditions
filtered_df["passes_lipinski"] = (
    # Hydrogen bond acceptors <= 10
    (filtered_df["hba_rdkit"] <= 10) &
    
    # Hydrogen bond donors <= 5
    (filtered_df["hbd_rdkit"] <= 5) &
    
    # LogP (lipophilicity) <= 5
    (filtered_df["logp_rdkit"] <= 5) &
    
    # Molecular weight <= 500
    (filtered_df["mw_rdkit"] <= 500)
)

# Print a preview of selected columns to inspect filtering results
print(
    filtered_df[
        [
            "chembl_id", # Unique ChEMBL identifier
            "pref_name", # Molecule name
            "canonical_smiles_rdkit", # Cleaned SMILES string
            "mw_rdkit", # Molecular weight (RDKit calculated)
            "passes_mw_filter", # Boolean: passes MW range filter
            "passes_lipinski" # Boolean: passes Lipinski rules
        ]
    ].head()
)

   chembl_id    pref_name                             canonical_smiles_rdkit  \
0   CHEMBL57   NEVIRAPINE                    Cc1ccnc2c1NC(=O)c1cccnc1N2C1CC1   
1  CHEMBL114   SAQUINAVIR  CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C...   
2  CHEMBL584   NELFINAVIR  Cc1c(O)cccc1C(=O)N[C@@H](CSc1ccccc1)[C@H](O)CN...   
3  CHEMBL116   AMPRENAVIR  CC(C)CN(C[C@@H](O)[C@H](Cc1ccccc1)NC(=O)O[C@H]...   
4  CHEMBL593  DELAVIRDINE  CC(C)Nc1cccnc1N1CCN(C(=O)c2cc3cc(NS(C)(=O)=O)c...   

   mw_rdkit  passes_mw_filter  passes_lipinski  
0   266.304              True             True  
1   670.855              True            False  
2   567.796              True            False  
3   505.637              True            False  
4   456.572              True             True  


### 4) Pull mechanism-of-action data for the J05 molecules

Adds mechanism to the training data (for example, polymerase inhibitors, reverse transcriptase inhibitors, protease inhibitors, etc.).

In [11]:
j05_ids = set(filtered_df["chembl_id"].dropna().unique())

mech_rows = []
for mech in mechanism.filter():
    chembl_id = mech.get("molecule_chembl_id")
    if chembl_id in j05_ids:
        mech_rows.append({
            "chembl_id": chembl_id,
            "target_chembl_id": mech.get("target_chembl_id"),
            "action_type": mech.get("action_type"),
            "mechanism_of_action": mech.get("mechanism_of_action"),
            "direct_interaction": mech.get("direct_interaction"),
            "disease_efficacy": mech.get("disease_efficacy")
        })

mech_df = pd.DataFrame(mech_rows).drop_duplicates()
print(mech_df.shape)
mech_df.head()

(67, 6)


,chembl_id,target_chembl_id,action_type,mechanism_of_action,direct_interaction,disease_efficacy
0,CHEMBL1643,CHEMBL1822,INHIBITOR,Inosine-5'-monophosphate dehydrogenase 1 inhib...,1,1
1,CHEMBL256907,CHEMBL274,ANTAGONIST,C-C chemokine receptor type 5 antagonist,1,1
2,CHEMBL853,CHEMBL247,INHIBITOR,Human immunodeficiency virus type 1 reverse tr...,1,1
3,CHEMBL1460,CHEMBL247,INHIBITOR,Human immunodeficiency virus type 1 reverse tr...,1,1
4,CHEMBL885,CHEMBL247,INHIBITOR,Human immunodeficiency virus type 1 reverse tr...,1,1


### 5) Create datasets for the rest of the pipeline

In [13]:
# Raw J05 pull
raw_j05_df = raw_df.copy()

# Cleaned molecular table
clean_j05_df = filtered_df.copy()

# Model training table: one row per molecule with the canonical SMILES you will train on
train_smiles_df = clean_j05_df[[
    "chembl_id",
    "pref_name",
    "canonical_smiles_rdkit",
    "mw_rdkit",
    "logp_rdkit",
    "hba_rdkit",
    "hbd_rdkit",
    "rotatable_bonds",
    "tpsa",
    "passes_mw_filter",
    "passes_lipinski"
]].rename(columns={"canonical_smiles_rdkit": "smiles"}).copy()

# Seed/reference molecules you to use in the app or evaluation
seed_names = {"REMDESIVIR", "MOLNUPIRAVIR", "RIBAVIRIN", "SOFOSBUVIR", "TENOFOVIR", "LAMIVUDINE", "EMTRICITABINE"}
seed_df = clean_j05_df[
    clean_j05_df["pref_name"].fillna("").str.upper().isin(seed_names)
].copy()

print("raw_j05_df:", raw_j05_df.shape)
print("clean_j05_df:", clean_j05_df.shape)
print("train_smiles_df:", train_smiles_df.shape)
print("seed_df:", seed_df.shape)

raw_j05_df: (89, 14)
clean_j05_df: (87, 24)
train_smiles_df: (87, 11)
seed_df: (6, 24)


In [14]:
# Save outputs for the rest of your pipeline
raw_j05_df.to_csv("raw_j05_antivirals.csv", index=False)
clean_j05_df.to_csv("clean_j05_antivirals.csv", index=False)
train_smiles_df.to_csv("train_smiles_j05.csv", index=False)
mech_df.to_csv("j05_mechanisms.csv", index=False)
seed_df.to_csv("seed_molecules_j05.csv", index=False)

print("Saved:")
print("- raw_j05_antivirals.csv")
print("- clean_j05_antivirals.csv")
print("- train_smiles_j05.csv")
print("- j05_mechanisms.csv")
print("- seed_molecules_j05.csv")

Saved:
- raw_j05_antivirals.csv
- clean_j05_antivirals.csv
- train_smiles_j05.csv
- j05_mechanisms.csv
- seed_molecules_j05.csv


### What these files are for

- **raw_j05_antivirals.csv**: original ChEMBL J05 pull
- **clean_j05_antivirals.csv**: validated and descriptor-enriched molecule table
- **train_smiles_j05.csv**: the main training file for the generative model
- **j05_mechanisms.csv**: mechanism-of-action annotations for subgrouping and analysis
- **seed_molecules_j05.csv**: recognizable antivirals 


### 6) Tokenize SMILES

In [15]:
import re

# SMILES tokenizer (handles atoms, brackets, bonds, rings, etc.) ---
SMILES_REGEX = r"(\[[^\[\]]+\]|Br?|Cl?|N|O|S|P|F|I|b|c|n|o|s|p|\(|\)|\.|=|#|-|\+|\\|\/|:|~|@|\?|>|\*|\$|\%[0-9]{2}|[0-9])"

def tokenize_smiles(smiles):
    """
    Tokenizes a SMILES string into a list of tokens.
    """
    tokens = re.findall(SMILES_REGEX, smiles)
    return tokens

In [16]:
# Apply tokenization
tokenized_smiles = filtered_df["canonical_smiles_rdkit"].dropna().apply(tokenize_smiles)

# Preview
print("Example tokenized SMILES:")
for i in range(3):
    print(tokenized_smiles.iloc[i])

Example tokenized SMILES:
['C', 'c', '1', 'c', 'c', 'n', 'c', '2', 'c', '1', 'N', 'C', '(', '=', 'O', ')', 'c', '1', 'c', 'c', 'c', 'n', 'c', '1', 'N', '2', 'C', '1', 'C', 'C', '1']
['C', 'C', '(', 'C', ')', '(', 'C', ')', 'N', 'C', '(', '=', 'O', ')', '[C@@H]', '1', 'C', '[C@@H]', '2', 'C', 'C', 'C', 'C', '[C@@H]', '2', 'C', 'N', '1', 'C', '[C@@H]', '(', 'O', ')', '[C@H]', '(', 'C', 'c', '1', 'c', 'c', 'c', 'c', 'c', '1', ')', 'N', 'C', '(', '=', 'O', ')', '[C@H]', '(', 'C', 'C', '(', 'N', ')', '=', 'O', ')', 'N', 'C', '(', '=', 'O', ')', 'c', '1', 'c', 'c', 'c', '2', 'c', 'c', 'c', 'c', 'c', '2', 'n', '1']
['C', 'c', '1', 'c', '(', 'O', ')', 'c', 'c', 'c', 'c', '1', 'C', '(', '=', 'O', ')', 'N', '[C@@H]', '(', 'C', 'S', 'c', '1', 'c', 'c', 'c', 'c', 'c', '1', ')', '[C@H]', '(', 'O', ')', 'C', 'N', '1', 'C', '[C@H]', '2', 'C', 'C', 'C', 'C', '[C@H]', '2', 'C', '[C@H]', '1', 'C', '(', '=', 'O', ')', 'N', 'C', '(', 'C', ')', '(', 'C', ')', 'C']


In [17]:
# Build vocabulary
all_tokens = set(token for tokens in tokenized_smiles for token in tokens)

# Add special tokens 
special_tokens = ["<PAD>", "<START>", "<END>"]
vocab = sorted(all_tokens) + special_tokens

# Create mappings
token_to_idx = {token: i for i, token in enumerate(vocab)}
idx_to_token = {i: token for token, i in token_to_idx.items()}

print(f"\nVocabulary size: {len(vocab)}")


Vocabulary size: 40


In [18]:
# Convert tokenized SMILES to integer sequences
def encode_smiles(tokens):
    return [token_to_idx["<START>"]] + \
           [token_to_idx[t] for t in tokens if t in token_to_idx] + \
           [token_to_idx["<END>"]]

encoded_smiles = tokenized_smiles.apply(encode_smiles)

# Preview encoded sequences
print("\nExample encoded SMILES:")
print(encoded_smiles.iloc[0])


Example encoded SMILES:
[38, 14, 33, 6, 33, 33, 34, 33, 7, 33, 6, 18, 14, 1, 12, 19, 2, 33, 6, 33, 33, 33, 34, 33, 6, 18, 7, 14, 6, 14, 14, 6, 39]


In [19]:
# Save outputs for training
filtered_df["tokenized_smiles"] = tokenized_smiles
filtered_df["encoded_smiles"] = encoded_smiles

# Save vocab and dataset
import json

with open("smiles_vocab.json", "w") as f:
    json.dump(token_to_idx, f)

filtered_df.to_csv("j05_tokenized_smiles.csv", index=False)

print("\nSaved:")
print("- smiles_vocab.json")
print("- j05_tokenized_smiles.csv")


Saved:
- smiles_vocab.json
- j05_tokenized_smiles.csv
